# frame_to_video —— 图片序列合成 GIF / MP4

把一个文件夹里的图片帧（如 `Frame_00000.png` …）合成动画。

**用法**：改下面【全局参数】单元格 → 从上到下逐格运行（或 `运行全部`）。

**依赖**：
- GIF：只需 Pillow（PIL），开箱即用。
- MP4：需 `imageio[ffmpeg]`。没装时脚本自动退回 GIF，不会报错。

In [ ]:
# ============================================================
# 【全局参数】—— 改这里就行
# ============================================================
import sys
from pathlib import Path

# Windows 终端默认 cp950/gbk，打印简体中文会报 UnicodeEncodeError，强制 UTF-8
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

# 输入：存放图片帧的文件夹
INPUT_DIR = r"C:\Users\GAI\Downloads\frames"

# 输出文件夹：默认 None = 跟 frame 文件夹同一处（即 INPUT_DIR）
# 想另存别处就改成 r"D:\xxx" 之类的路径
OUTPUT_DIR = None

# 输出文件名（不含后缀，后缀按 OUTPUT_FORMAT 自动加）
OUTPUT_NAME = "frames_out"

# 输出格式："gif" 或 "mp4"
OUTPUT_FORMAT = "gif"

# 帧率：每秒多少张图（越大播放越快）
FPS = 12

# 匹配哪些图片（按文件名排序保证帧顺序）。默认 png
PATTERN = "*.png"

# 缩放：把每帧宽度缩到此像素（高等比）。None = 不缩放
# GIF 体积大，建议 800~1000；MP4 可设 None 保原尺寸
RESIZE_WIDTH = None

# GIF 是否循环：0 = 无限循环；其它数字 = 循环几次
GIF_LOOP = 0

In [ ]:
# ============================================================
# 计算实际输出路径：OUTPUT_DIR 为 None 时落在 frame 文件夹里
# ============================================================
out_dir = Path(OUTPUT_DIR) if OUTPUT_DIR else Path(INPUT_DIR)  # 默认 = frame 文件夹
out_dir.mkdir(parents=True, exist_ok=True)                    # 不存在就建
OUTPUT_PATH = out_dir / OUTPUT_NAME                           # 拼成 文件夹/文件名（无后缀）
print(f"[信息] 输出文件夹 → {out_dir}")

In [ ]:
# ============================================================
# 工具函数
# ============================================================
def collect_frames(input_dir, pattern):
    """收集并按文件名排序所有帧路径，返回 list[Path]。"""
    files = sorted(Path(input_dir).glob(pattern))  # glob + 排序保证 00,01,02… 顺序
    if not files:
        raise FileNotFoundError(f"在 {input_dir} 找不到匹配 {pattern} 的图片")
    print(f"[信息] 找到 {len(files)} 帧")
    return files


def load_resized(path, width):
    """读取一张图；若指定宽度则等比缩放。返回 PIL.Image。"""
    from PIL import Image
    img = Image.open(path).convert("RGB")  # 统一转 RGB，避免调色板/透明通道问题
    if width:                              # 需要缩放时按宽度等比算高度
        h = int(img.height * width / img.width)
        img = img.resize((width, h), Image.LANCZOS)  # LANCZOS：高质量缩放
    return img


def make_gif(files, out_path, fps, width, loop):
    """用 Pillow 合成 GIF。返回生成文件路径。"""
    out_path = Path(out_path).with_suffix(".gif")
    duration = int(1000 / fps)                        # GIF 每帧毫秒数 = 1000/帧率
    frames = [load_resized(f, width) for f in files]  # 全部读进内存
    frames[0].save(                                   # 第一帧带头，append 接后续
        out_path, save_all=True, append_images=frames[1:],
        duration=duration, loop=loop, optimize=True,  # optimize 压缩调色板减体积
    )
    print(f"[完成] GIF 已生成 → {out_path}")
    return out_path


def make_mp4(files, out_path, fps, width):
    """用 imageio 合成 MP4。缺 imageio/ffmpeg 会抛 ImportError，由调用方退回 GIF。"""
    import numpy as np                     # imageio 写帧需要 ndarray
    import imageio.v2 as imageio           # 缺则 ImportError → 触发退回
    out_path = Path(out_path).with_suffix(".mp4")
    writer = imageio.get_writer(out_path, fps=fps, codec="libx264")
    for f in files:
        writer.append_data(np.asarray(load_resized(f, width)))  # PIL → ndarray 写入
    writer.close()
    print(f"[完成] MP4 已生成 → {out_path}")
    return out_path

In [ ]:
# ============================================================
# 执行合成
# ============================================================
files = collect_frames(INPUT_DIR, PATTERN)

if OUTPUT_FORMAT.lower() == "mp4":
    try:
        result_path = make_mp4(files, OUTPUT_PATH, FPS, RESIZE_WIDTH)
    except ImportError:
        # 缺 imageio/ffmpeg：不中断，自动退回 GIF
        print("[警告] 缺少 imageio 或 ffmpeg，无法输出 MP4，自动改输出 GIF。")
        print("       想要 MP4 请先装： pip install imageio[ffmpeg]")
        result_path = make_gif(files, OUTPUT_PATH, FPS, RESIZE_WIDTH, GIF_LOOP)
else:
    result_path = make_gif(files, OUTPUT_PATH, FPS, RESIZE_WIDTH, GIF_LOOP)

In [ ]:
# ============================================================
# 预览结果（GIF 直接在 notebook 内播放；MP4 给出路径）
# ============================================================
from IPython.display import Image as IPyImage, Video, display

if str(result_path).lower().endswith(".gif"):
    display(IPyImage(filename=str(result_path)))  # 内嵌播放 GIF
else:
    display(Video(str(result_path), embed=True))  # 内嵌播放 MP4